In [1]:
import torch
print(torch.__version__, torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime -> Change runtime type -> T4 GPU")
print(torch.cuda.get_device_name(0))

2.11.0+cu128 True
Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
ZIP = '/content/drive/MyDrive/submission.zip'

!rm -rf /content/submission
!unzip -q "$ZIP" -d /content/
%cd /content/submission
!ls

/content/submission
check_leakage.py	evaluate_pairs.py	README.md
colab_train.ipynb	gallery_probe.py	report.pdf
dataset			generate_pairs.py	requirements.txt
dataset_preparation.py	identity_manifest.json	roc_analysis.py
dataset_stats.json	make_report.py		splits.json
demo_api.py		model.py		train.py


In [4]:
import json
splits = json.load(open('splits.json'))
for k, v in splits.items():
    print(k, len(v), 'identities')

overlap = set(splits['train']) & (set(splits['val']) | set(splits['test']))
print('train/eval overlap:', len(overlap))
assert not overlap

train 1500 identities
val 60 identities
test 120 identities
train/eval overlap: 0


In [7]:
!python train.py --root . --size 224 --epochs 30 \
    --batch-p 32 --batch-k 4 --lr 1e-3 --workers 2

device=cuda
7096 images / 1500 identities, 55 batches of 128
[  1/30] loss=20.2796 (arc=19.8150 tri=0.4646) acc=0.001 | val AUC=0.8129 EER=0.2625 | 40s
        saved (val AUC 0.8129)
[  2/30] loss=18.2537 (arc=17.8599 tri=0.3938) acc=0.010 | val AUC=0.8318 EER=0.2480 | 40s
        saved (val AUC 0.8318)
[  3/30] loss=16.9690 (arc=16.6095 tri=0.3595) acc=0.009 | val AUC=0.8431 EER=0.2485 | 41s
        saved (val AUC 0.8431)
[  4/30] loss=14.7030 (arc=14.3858 tri=0.3172) acc=0.023 | val AUC=0.9005 EER=0.1858 | 41s
        saved (val AUC 0.9005)
[  5/30] loss=12.6157 (arc=12.3414 tri=0.2743) acc=0.044 | val AUC=0.9005 EER=0.1820 | 40s
        saved (val AUC 0.9005)
[  6/30] loss=10.8010 (arc=10.5642 tri=0.2368) acc=0.083 | val AUC=0.9115 EER=0.1763 | 40s
        saved (val AUC 0.9115)
[  7/30] loss=9.2343 (arc=9.0324 tri=0.2020) acc=0.132 | val AUC=0.9206 EER=0.1605 | 41s
        saved (val AUC 0.9206)
[  8/30] loss=7.9717 (arc=7.8008 tri=0.1708) acc=0.191 | val AUC=0.9253 EER=0.1583 | 41

In [8]:
import json
h = json.load(open('checkpoints/training_history.json'))
for r in h['history']:
    print(f"epoch {r['epoch']:3d}  loss {r['loss']:.4f}  acc {r['train_acc']:.3f}  "
          f"val AUC {r['val_auc']:.4f}  val EER {r['val_eer']:.4f}")
print('best val AUC:', h['best_val_auc'])

epoch   1  loss 20.2796  acc 0.001  val AUC 0.8129  val EER 0.2625
epoch   2  loss 18.2537  acc 0.010  val AUC 0.8318  val EER 0.2480
epoch   3  loss 16.9690  acc 0.009  val AUC 0.8431  val EER 0.2485
epoch   4  loss 14.7030  acc 0.023  val AUC 0.9005  val EER 0.1858
epoch   5  loss 12.6157  acc 0.044  val AUC 0.9005  val EER 0.1820
epoch   6  loss 10.8010  acc 0.083  val AUC 0.9115  val EER 0.1763
epoch   7  loss 9.2343  acc 0.132  val AUC 0.9206  val EER 0.1605
epoch   8  loss 7.9717  acc 0.191  val AUC 0.9253  val EER 0.1583
epoch   9  loss 6.8800  acc 0.260  val AUC 0.9342  val EER 0.1460
epoch  10  loss 6.1169  acc 0.325  val AUC 0.9447  val EER 0.1260
epoch  11  loss 5.5537  acc 0.383  val AUC 0.9373  val EER 0.1405
epoch  12  loss 4.9671  acc 0.449  val AUC 0.9389  val EER 0.1333
epoch  13  loss 4.4321  acc 0.511  val AUC 0.9502  val EER 0.1215
epoch  14  loss 4.0665  acc 0.552  val AUC 0.9513  val EER 0.1180
epoch  15  loss 3.6682  acc 0.600  val AUC 0.9501  val EER 0.1197
epoc

In [10]:
!cp checkpoints/best_model.pth checkpoints/training_history.json /content/drive/MyDrive/

from google.colab import files
files.download('checkpoints/best_model.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>